<a href="https://colab.research.google.com/github/RomanCarrubba/Econ-5200-Problem-Set-1/blob/main/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np


def simulate_transactions(n=20000, seed=42):
    """
    returns a dataframe with simulated transactions
    """
    np.random.seed(seed)

    year = np.random.choice([2020, 2021], size=n)

    cancelled = np.where(
        year == 2021,
        np.random.choice([0, 1], size=n, p=[0.9, 0.1]),
        0
    )

    order_type = np.array(["B2C"] * n)

    b2b_indices = np.random.choice(n, size=20, replace=False)
    order_type[b2b_indices] = "B2B"

    basket_value = np.random.lognormal(
        mean=3.5,
        sigma=0.8,
        size=n
    )

    basket_value[b2b_indices] *= 100

    data = {
        "basket_value": basket_value,
        "year": year,
        "order_type": order_type,
        "customer_id": np.random.choice(1000, size=n),
        "cancelled": cancelled,
    }

    df = pd.DataFrame(data)

    return df

def compute_naive_mean(df):
    """
    returns a list with the mean basket value for 2020 and 2021
    """
    year2021 = df.loc[df["year"] == 2021]
    year2020 = df.loc[df["year"] == 2020]

    return [
        year2020["basket_value"].mean(),
        year2021["basket_value"].mean()
    ]

def compute_true_mean(df):
    """
    returns a list with the mean basket value for 2020 and 2021
    """
    true_2020 = df.loc[
        (df["year"] == 2020) & (df["order_type"] == "B2C") & (df["cancelled"] == 0),
        "basket_value"
    ].mean()

    true_2021 = df.loc[
        (df["year"] == 2021) & (df["order_type"] == "B2C") & (df["cancelled"] == 0),
        "basket_value"
    ].mean()
    return [true_2020, true_2021]

def compute_mean_gap(df):
  """
  returns a list with the gap between true and naive mean basket value for 2020 and 2021
  """
  true_mean = compute_true_mean(df)
  naive_mean = compute_naive_mean(df)
  return [np.abs(true_mean[0] - naive_mean[0]), np.abs(true_mean[1] - naive_mean[1])]

df = simulate_transactions()
naive_mean = compute_naive_mean(df)
true_mean = compute_true_mean(df)
mean_gap = compute_mean_gap(df)
print(f"The naive mean of 2020 and 2021: {naive_mean[0]}, {naive_mean[1]}")
print(f"The true mean of 2020 and 2021: {true_mean[0]}, {true_mean[1]}")
print(f"The gap between the true and naive mean for 2020 and 2021: {mean_gap[0]}, {mean_gap[1]}")



The naive mean of 2020 and 2021: 49.3380734922399, 51.60815246087797
The true mean of 2020 and 2021: 45.97798987067258, 45.71762641987248
The gap between the true and naive mean for 2020 and 2021: 3.3600836215673198, 5.8905260410054865


In [2]:
def find_effects_of_contamination(df):
    """
    returns the contribution of B2B contamination and
    logging changes to the naive mean for 2020 and 2021.
    """

    true_mean = compute_true_mean(df)
    naive_mean = compute_naive_mean(df)

    b2b_only_2020 = df.loc[
        (df["year"] == 2020) & (df["cancelled"] == 0),
        "basket_value"
    ].mean()

    b2b_only_2021 = df.loc[
        (df["year"] == 2021) & (df["cancelled"] == 0),
        "basket_value"
    ].mean()

    b2b_effect = [
        b2b_only_2020 - true_mean[0],
        b2b_only_2021 - true_mean[1]
    ]

    logging_effect = [
        naive_mean[0] - b2b_only_2020,
        naive_mean[1] - b2b_only_2021
    ]

    return [b2b_effect, logging_effect]

"""
median
trimmed mean
mean agter b2b exlusion
"""

def find_median(df):
    """
    returns the median of the basket value for 2020 and 2021
    """
    year2021 = df.loc[df["year"] == 2021]
    year2020 = df.loc[df["year"] == 2020]
    return [year2020["basket_value"].median(), year2021["basket_value"].median()]

def find_trimmed_mean(df, trim):
    """
    Returns the trimmed mean of basket value for 2020 and 2021.
    """

    year2021 = df.loc[df["year"] == 2021]
    year2020 = df.loc[df["year"] == 2020]

    years = [year2020, year2021]
    result = []

    for year in years:
        sorted_data = sorted(year["basket_value"])
        n = len(sorted_data)

        k = int(n * trim)

        if k > 0:
            remaining_data = sorted_data[k:-k]
        else:
            remaining_data = sorted_data

        if not remaining_data:
            result.append(0)
        else:
            result.append(sum(remaining_data) / len(remaining_data))

    return result

def find_mean_after_b2b_exclusion(df):
    """
    returns the mean of basket value for 2020 and 2021 after B2B contamination.
    """
    without_b2b_2020 = df.loc[
        (df["year"] == 2020) & (df["order_type"] == "B2C"),
        "basket_value"
    ].mean()

    without_b2b_2021 = df.loc[
        (df["year"] == 2021) & (df["order_type"] == "B2C"),
        "basket_value"
    ].mean()
    return [without_b2b_2020, without_b2b_2021]



print(f"The results of contamination: {find_effects_of_contamination(df)}")
print(f"The median basket value for 2020 and 2021: {find_median(df)[0]}, {find_median(df)[1]}")
print(f"The 10% trimmed mean for 2020 and 2021: {find_trimmed_mean(df, 0.1)[0]}, {find_trimmed_mean(df, 0.1)[1]}")
print(f"The mean after B2B exclusion for 2020 and 2021: {find_mean_after_b2b_exclusion(df)[0]}, {find_mean_after_b2b_exclusion(df)[1]}")

The results of contamination: [[np.float64(3.3600836215673198), np.float64(6.2983883259727875)], [np.float64(0.0), np.float64(-0.40786228496730104)]]
The median basket value for 2020 and 2021: 33.28938209267342, 33.38633255431257
The 10% trimmed mean for 2020 and 2021: 38.457619507128555, 38.32621117129096
The mean after B2B exclusion for 2020 and 2021: 45.97798987067258, 45.74759576489542


####Which number should the dashboard show?

I would recommend the **mean after B2B exclusion**. This is the most accurate measure of the consumer basket because it gets rid of the massive B2B sales that weren't meant to be in the dataset in the first place. It also doesn't remove real consumer data points like the trimmed mean could. This makes the result easier to explain and ensures the dashboard is measuring what it's supposed to.

In [3]:
%%writefile basket_metrics.py

import pandas as pd
import numpy as np

def audit_report(df: pd.DataFrame) -> pd.DataFrame:
  """
  Returns a dataframe with the audit report.
  """
  result = []
  for column in df:
    series = df[column]

    missing = series.isna().mean()

    dtype = series.dtype

    if (pd.api.types.is_numeric_dtype(series)):
        skew = series.skew()

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = (
            (series < lower_bound) |
            (series > upper_bound)
        ).sum()

    else:
        skew = np.nan
        outlier_count = np.nan

    result.append({
        "column": column,
        "missing": missing,
        "dtype": dtype,
        "skew": skew,
        "outlier_count": outlier_count
    })
  return pd.DataFrame(result)

def robust_mean(
    x: pd.Series,
    method: str,
    order_type: pd.Series| None = None
    ) -> float:
  """
  Returns median, trimmed or rule-based exclusion
  """
  match method:
    case "median":
      return x.median()

    case "trimmed":
      lower = x.quantile(0.1)
      upper = x.quantile(0.9)

      return x.loc[
          (x >= lower) & (x <= upper)
      ].mean()

    case "rule-based":
      if order_type is None:
        raise ValueError("order_type must be provided")
      return x.loc[order_type == "B2C"].mean()

    case _:
            raise ValueError(
                "The method is not valid. "
                "Use 'median', 'trimmed', or 'rule-based'."
            )

if __name__ == "__main__":
    df = pd.DataFrame({
        "basket_value": [10, 12, 15, 20, 25, 30, 1000],
        "order_type": ["B2C", "B2C", "B2C", "B2C", "B2C", "B2C", "B2B"]
    })

    print("Audit Report:")
    print(audit_report(df))

    print("\nMedian:")
    print(robust_mean(df["basket_value"], "median"))

    print("\nTrimmed Mean:")
    print(robust_mean(df["basket_value"], "trimmed"))

    print("\nRule-based Exclusion:")
    print(robust_mean(df["basket_value"], "rule-based", df["order_type"]))




Writing basket_metrics.py


In [4]:
from typing import runtime_checkable
import basket_metrics as bm

test_df = pd.DataFrame({
    "basket_value": [10, 12, 15, 18, 20, 22, 25, 30, 35, 1000],
    "order_type": [
        "B2C", "B2C", "B2C", "B2C", "B2C",
        "B2C", "B2C", "B2C", "B2C", "B2B"
    ],
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
})

median = bm.robust_mean(test_df["basket_value"], "median")
trimmed = bm.robust_mean(test_df["basket_value"], "trimmed")
rule_based = bm.robust_mean(test_df["basket_value"], "rule-based", test_df["order_type"])

print("Audit Report:")
print(bm.audit_report(test_df))

print("robust_mean: median")
print(median)

print("robust_mean: trimmed")
print(trimmed)

print("robust_mean: rule based")
print(rule_based)

assert median == 21
assert rule_based == 20.77777777777778
assert trimmed < test_df["basket_value"].mean()

Audit Report:
         column  missing   dtype      skew  outlier_count
0  basket_value      0.0   int64  3.158567            1.0
1    order_type      0.0  object       NaN            NaN
2   customer_id      0.0   int64  0.000000            0.0
robust_mean: median
21.0
robust_mean: trimmed
22.125
robust_mean: rule based
20.77777777777778


##Strongest Objection

B2B is real revenue. Why are you removing our biggest buyers from the number we manage the business on?"

##The single analysis that most reduces exposure

Build a stepwise bridge for each year, from the naive mean to your recommended metric, with one correction added at a time: naive mean, B2B excluded only, B2B excluded on a logging-harmonized definition, then trimmed. Show each step's contribution to the level and to the year-over-year change, alongside the trim-vs-flag overlap table.


In [33]:
def step(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a stepwise bridge from the naive mean to the
    trimmed mean for each year.
    """

    result = []

    for year in [2020, 2021]:

        year_data = df.loc[df["year"] == year]

        # Step 1: naive mean
        naive_mean = year_data["basket_value"].mean()

        # Step 2: B2B excluded
        b2b_excluded = year_data.loc[
            year_data["order_type"] == "B2C",
            "basket_value"
        ].mean()

        # Step 3: B2B excluded and cancellations excluded
        logging_harmonized = year_data.loc[
            (year_data["order_type"] == "B2C") &
            (year_data["cancelled"] == 0),
            "basket_value"
        ].mean()

        # Step 4: 10% trimmed mean
        consumer_data = year_data.loc[
            (year_data["order_type"] == "B2C") &
            (year_data["cancelled"] == 0),
            "basket_value"
        ]

        lower = consumer_data.quantile(0.1)
        upper = consumer_data.quantile(0.9)

        trimmed = consumer_data.loc[
            (consumer_data >= lower) &
            (consumer_data <= upper)
        ].mean()

        b2b_contribution = naive_mean - b2b_excluded
        logging_contribution = b2b_excluded - logging_harmonized
        trimmed_contribution = logging_harmonized - trimmed

        result.append({
            "year": year,
            "naive_mean": naive_mean,
            "b2b_excluded": b2b_excluded,
            "b2b contribution": b2b_contribution,
            "logging harmonized": logging_harmonized,
            "logging contribution": logging_contribution,
            "trimmed": trimmed,
            "trim contribution": trimmed_contribution
        })

    return pd.DataFrame(result)


print(step(df))


   year  naive_mean  b2b_excluded  b2b contribution  logging harmonized  \
0  2020   49.338073     45.977990          3.360084           45.977990   
1  2021   51.608152     45.747596          5.860557           45.717626   

   logging contribution    trimmed  trim contribution  
0              0.000000  38.422878           7.555112  
1              0.029969  38.191135           7.526491  
10th percentile: 11.983722641359988
90th percentile: 94.13879632312515


####HEADLINE:
The dashboard overstated the consumer basket metric by 11 dollars in 2020 and 13 dollars in 2021.

####CORRECTION:
Removing B2B orders and harmonizing logging changes the 2021 mean from 51.61 dollars to 38.19 dollars .

####CONFIDENCE:
I have moderate confidence in this becuase while the number seems to be more centered and not effected by outliers, there was a lot of data that was taken away to get it.

####BEFORE BETTING ON IT:
I would want to confirm the data was input correctly and all B2B data and cancelations where labeled correctly.